# Metro-ASR — Fine-Tuning Guide

Adapt the released Metro-Small checkpoint to your own data — a new domain, accent, or
vocabulary — without training from scratch.

This notebook covers:
1. Downloading the pretrained checkpoint to fine-tune from
2. Fine-tuning on a HuggingFace dataset
3. Fine-tuning on your own local audio
4. Training a domain language head (text only, no audio, no GPU)
5. Training a custom BPE tokenizer
6. Testing the fine-tuned model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/metro-asr/blob/main/examples/fine_tuning.ipynb)

> **Requires a GPU.** Colab menu: *Runtime → Change runtime type → T4 GPU*.
> Section 4 (the language head) is the exception — it's CPU-only and takes minutes, not hours.


## Setup

In [ ]:
!pip install -q "metro-asr[train,lm]"   # [lm] is for section 4's beam_search=True
!pip install -q -U "numpy>=2.0"          # see note below
!git clone -q https://github.com/MohammedAly22/metro-asr.git
%cd metro-asr


> [!NOTE]
> `pyctcdecode` (needed for beam search) still declares `numpy<2.0.0` in its own
> package metadata, even though it runs fine under numpy 2.x — pip will honor that and
> downgrade numpy to satisfy it. On Colab, which has numpy 2.x pre-installed and several
> other packages (jax, opencv, scipy) already built against it, that downgrade breaks
> those packages' compiled extensions with a `numpy.dtype size changed` error the next
> time anything imports them. The fix is the second line below: reinstall numpy 2.x
> *after* metro-asr, which restores a consistent environment without touching
> `pyctcdecode` itself.

## 1. Download the pretrained checkpoint

Fine-tuning starts from the released weights, so get them first — this is the step it's
easiest to forget, and every fine-tuning command below depends on `checkpoints/` existing.
About 900 MB (weights + tokenizer); add `"*.bin"` to `allow_patterns` if you also want the
6 GB general-purpose language model, which fine-tuning itself doesn't need.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="MohammedAly22/metro-asr-small",
    local_dir="checkpoints",
    allow_patterns=["config.yaml", "model.pt", "bpe.model", "bpe.vocab"],
)
print("Downloaded to checkpoints/")


## 2. Fine-tune on a HuggingFace dataset

`scripts/finetune.py` takes CLI flags — nothing to hand-edit. Defaults: learning rate 10-20x
below pretraining, and the encoder frozen for the first few thousand steps so the CTC head
adapts to the new data before touching the acoustic representations (see the README's
[Fine-tuning](https://github.com/MohammedAly22/metro-asr#fine-tuning) section for why).

This runs for real — expect it to take a while on a T4. Lower `--max-steps` for a quick test.

In [ ]:
!python scripts/finetune.py \
    --config configs/metro_small.yaml \
    --tokenizer-dir checkpoints \
    --checkpoint checkpoints/model.pt \
    --dataset MohamedRashad/arabic-english-code-switching \
    --lr 5e-5 \
    --max-steps 30000 \
    --freeze-steps 3000 \
    --run-suffix cs-finetune


## 3. Fine-tune on your own local audio

First, package your clips as a HuggingFace `Dataset` with an `audio` and a `text`
column — the same format `scripts/prepare_data.py` produces, see the README's
[Data format](https://github.com/MohammedAly22/metro-asr#data-format) section for the full spec:

In [ ]:
from datasets import Dataset, Audio

# Replace with your own (path, transcript) pairs — dozens of examples is a lot better than a
# handful, but this format works at any size.
data = [
    {"audio": "path/to/audio1.wav", "text": "أنا رايح الـ meeting"},
    {"audio": "path/to/audio2.wav", "text": "الـ project ده محتاج update"},
]

dataset = Dataset.from_list(data).cast_column("audio", Audio(sampling_rate=16000))
split = dataset.train_test_split(test_size=0.1, seed=42)
split["train"].save_to_disk("my_data/train")
split["test"].save_to_disk("my_data/eval")
print(f"train={len(split['train'])}  eval={len(split['test'])}")


In [ ]:
!python scripts/finetune.py \
    --config configs/metro_small.yaml \
    --tokenizer-dir checkpoints \
    --checkpoint checkpoints/model.pt \
    --prepared-data my_data \
    --lr 5e-5 \
    --max-steps 20000 \
    --freeze-steps 3000 \
    --run-suffix local-finetune


## 4. Training a domain language head

The part of Metro-ASR that adapts in *minutes, on CPU, from text alone* — no audio, no GPU.
See [Training a language head only](https://github.com/MohammedAly22/metro-asr#training-a-language-head-only)
and [Domain-specialised heads](https://github.com/MohammedAly22/metro-asr#domain-specialised-heads)
in the README for the full picture, including a measured comparison against a mismatched head.

Point `--corpus` at any text file, one sentence per line — your domain's vocabulary, product
names, documentation, whatever the acoustic model will need to spell correctly.

In [ ]:
!python scripts/train_lm.py \
    --corpus corpora/my_domain.txt \
    --out lm/my_domain_4gram.arpa \
    --order 4


In [ ]:
from metro_asr import MetroASREngine

engine = MetroASREngine.from_pretrained("checkpoints", lm_path="lm/my_domain_4gram.arpa")
result = engine.transcribe("audio.wav", beam_search=True)
print(result.text)


## 5. Training a custom BPE tokenizer

Only needed if you're changing the vocabulary itself (e.g. scaling up to Medium/Large,
or retraining on a very different language mix) — most fine-tuning keeps the released
tokenizer as-is, from Section 2 or 3 above.

In [ ]:
!python scripts/train_bpe_tokenizer.py \
    --corpus corpora/my_domain.txt \
    --vocab-size 5000 \
    --out my_tokenizer


## Fine-tuning reference

| Parameter | Recommended | Notes |
|---|---|---|
| Learning rate | `1e-4` to `5e-5` | 10-20x lower than pretraining |
| Freeze steps | 3,000 – 10,000 | Lets the CTC head adapt before the encoder moves |
| Max steps | 20,000 – 50,000 | Depends on dataset size — watch eval WER, stop when it turns |
| Batch size | 16 – 32 | Set in the config's `training.batch_size` |
| SpecAugment | keep enabled | Helps most on small datasets |

Monitor with Weights & Biases by setting `WANDB_PROJECT` before launching:

```bash
WANDB_PROJECT=metro-finetune python scripts/finetune.py ...
```

## 6. Test the fine-tuned model

In [ ]:
import torch
from metro_asr import MetroASREngine

engine = MetroASREngine.from_local(
    config_path="configs/metro_small.yaml",
    checkpoint_path="checkpoints/metro-small-cs-finetune/best_model.pt",
    tokenizer_dir="checkpoints",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

result = engine.transcribe("audio.wav")
print(f"Text: {result.text}")
print(f"RTF:  {result.rtf:.4f}")
